# Export images out of h5 format to folder

In [ ]:
# Imports
import h5py
from PIL import Image
import numpy as np
import glob
import os

h5_path = r"path\to\your\input_dataset.h5"                       # Change this to the path of your input dataset
out_root = r"path\to\your\output_directory\name_of_output.png"   # Change this to your desired output directory and name

os.makedirs(out_root, exist_ok=True)

with h5py.File(h5_path, "r") as f:
    for fold in f.keys():
        fold_dir = os.path.join(out_root, fold)
        os.makedirs(fold_dir, exist_ok=True)

        images = f[fold]["image"][:]   # adjust if name differs

        for i, img in enumerate(images):
            if img.dtype == np.float32 or img.dtype == np.float64:
                if img.max() <= 1.0:
                    img = (img * 255).astype(np.uint8)
                else:
                    img = img.astype(np.uint8)
            
            # Remove single channel dimension if present: (224, 224, 1) -> (224, 224)
            if len(img.shape) == 3 and img.shape[2] == 1:
                img = img.squeeze(axis=2)
            
            Image.fromarray(img).save(
                os.path.join(fold_dir, f"{fold}_{i:04d}.png")
            )

# Put back in h5 format and keep metadata

In [ ]:
import h5py
from PIL import Image
import numpy as np
import glob
import os

in_root = r"path\to\your\input_images_directory.png"        # Change this to the path of your input images directory
original_h5 = r"path\to\your\original_dataset.h5"           # Change this to the path of your original dataset
out_h5 = r"path\to\your\output_dataset.h5"                  # Change this to desired output path and name
img_size = (800, 800)                                       # Change this to image size (width, height)

# Open original file to read metadata
with h5py.File(original_h5, "r") as f_orig:
    with h5py.File(out_h5, "w") as f_out:
        for fold in sorted(os.listdir(in_root)):
            fold_path = os.path.join(in_root, fold)
            files = sorted(glob.glob(f"{fold_path}/*.png"))
            
            # Load and normalize images
            loaded_images = []
            for p in files:
                img = Image.open(p)
                
                # Force resize to desired size if needed
                if img.size != img_size:
                    img = img.resize(img_size, Image.LANCZOS)
                
                # Convert to numpy array
                img_array = np.array(img)
                
                # Ensure grayscale format (img_size) - remove extra dimensions
                if len(img_array.shape) == 3:
                    if img_array.shape[2] == 3:  # RGB to grayscale
                        img_array = np.mean(img_array, axis=2).astype(img_array.dtype)
                    elif img_array.shape[2] == 1:  # (img_size, 1) to (img_size)
                        img_array = img_array.squeeze(axis=2)
                
                loaded_images.append(img_array)
            
            # Verify all shapes match before stacking
            shapes = [img.shape for img in loaded_images]
            if len(set(shapes)) > 1:
                print(f"Shape mismatch in {fold}: {set(shapes)}")
                continue
            
            images = np.stack(loaded_images)
            
            # Create group
            grp = f_out.create_group(fold)
            
            # Save images
            grp.create_dataset("image", data=images, compression="gzip")
            
            # Copy all other datasets (target, diagnosis, etc.)
            if fold in f_orig:
                for key in f_orig[fold].keys():
                    if key != "image":  # Skip images, we already saved them
                        grp.create_dataset(key, data=f_orig[fold][key][:])
                
                # Copy attributes if any
                for attr_name, attr_value in f_orig[fold].attrs.items():
                    grp.attrs[attr_name] = attr_value
            
            print(f"Processed {fold}: {images.shape}")

print("All images saved with metadata preserved!")